In [ ]:

# CineMood - Intelligent Movie Recommendation System
# Assignment 3 - Harnessing AI for Business
# Section 1: Environment Setup

!pip -q install requests pandas numpy scikit-learn nltk plotly streamlit


In [ ]:

# Section 2: Import Libraries

import requests
import pandas as pd
import numpy as np
import nltk
import plotly.express as px

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.sentiment import SentimentIntensityAnalyzer
from google.colab import userdata

# Download NLP resources
nltk.download("vader_lexicon", quiet=True)

sia = SentimentIntensityAnalyzer()

print("All libraries loaded successfully!")


In [ ]:

# Section 3: TMDB API Configuration

TMDB_TOKEN = userdata.get("TMDB_TOKEN")

BASE_URL = "https://api.themoviedb.org/3"
IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"

HEADERS = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_TOKEN}"
}

def tmdb_request(endpoint, params=None):
    """
    Retrieve data from the TMDB API.
    Includes timeout and HTTP error handling.
    """
    response = requests.get(
        f"{BASE_URL}/{endpoint.lstrip('/')}",
        headers=HEADERS,
        params=params,
        timeout=30
    )

    response.raise_for_status()
    return response.json()


# Test API connection
try:
    configuration = tmdb_request("configuration")
    print("TMDB API connected successfully!")

except requests.exceptions.RequestException as error:
    print(f"Connection failed: {error}")


In [ ]:

# Section 4: Initial Movie Data Retrieval

data = tmdb_request(
    "movie/popular",
    params={
        "language": "en-US",
        "page": 1
    }
)

movies = pd.DataFrame(data["results"])

columns = [
    "id",
    "title",
    "release_date",
    "overview",
    "vote_average",
    "vote_count",
    "popularity",
    "original_language"
]

movies = movies[columns]

print(f"Movies retrieved: {len(movies)}")

display(movies.head(10))


In [ ]:

# Section 5: Movie Catalog Collection

from datetime import date

all_movies = []

# Collect popular and top-rated movies
for endpoint in ["movie/popular", "movie/top_rated"]:
    for page in range(1, 16):
        data = tmdb_request(
            endpoint,
            params={
                "language": "en-US",
                "page": page
            }
        )
        all_movies.extend(data.get("results", []))

# Collect movies from different decades
decades = [1970, 1980, 1990, 2000, 2010, 2020]

for decade in decades:
    for page in range(1, 6):
        data = tmdb_request(
            "discover/movie",
            params={
                "language": "en-US",
                "sort_by": "popularity.desc",
                "primary_release_date.gte": f"{decade}-01-01",
                "primary_release_date.lte": min(
                    f"{decade + 9}-12-31",
                    date.today().isoformat()
                ),
                "vote_count.gte": 50,
                "include_adult": "false",
                "page": page
            }
        )
        all_movies.extend(data.get("results", []))

# Remove duplicate movies
movies_df = pd.DataFrame(all_movies)
movies_df = movies_df.drop_duplicates(
    subset="id"
).reset_index(drop=True)

# Exclude unreleased movies and missing descriptions
movies_df["release_date"] = pd.to_datetime(
    movies_df["release_date"], errors="coerce"
)

movies_df = movies_df[
    (movies_df["release_date"] <= pd.Timestamp.today())
    & movies_df["overview"].fillna("").str.strip().ne("")
].reset_index(drop=True)

print(f"Total unique movies: {len(movies_df)}")

display(movies_df[[
    "id", "title", "release_date",
    "vote_average", "vote_count"
]].head(10))


In [ ]:

# Section 6: Detailed Movie Metadata

from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def get_movie_details(movie_id):

    data = tmdb_request(
        f"movie/{movie_id}",
        params={
            "language": "en-US",
            "append_to_response":
                "credits,keywords,release_dates"
        }
    )

    credits = data.get("credits", {})

    cast = [
        person["name"]
        for person in credits.get("cast", [])[:10]
    ]

    directors = [
        person["name"]
        for person in credits.get("crew", [])
        if person.get("job") == "Director"
    ]

    genres = [
        genre["name"]
        for genre in data.get("genres", [])
    ]

    keywords = [
        keyword["name"]
        for keyword in data.get(
            "keywords", {}
        ).get("keywords", [])
    ]

    # Retrieve US age certification
    certification = "Not Rated"

    for country in data.get(
        "release_dates", {}
    ).get("results", []):

        if country["iso_3166_1"] == "US":
            for release in country["release_dates"]:
                if release.get("certification"):
                    certification = release["certification"]
                    break
            break

    return {
        "id": data["id"],
        "title": data.get("title", ""),
        "release_date": data.get("release_date", ""),
        "overview": data.get("overview", ""),
        "genres": genres,
        "keywords": keywords,
        "cast": cast,
        "directors": directors,
        "runtime": data.get("runtime"),
        "certification": certification,
        "original_language": data.get("original_language"),
        "vote_average": data.get("vote_average"),
        "vote_count": data.get("vote_count"),
        "popularity": data.get("popularity"),
        "budget": data.get("budget"),
        "revenue": data.get("revenue"),
        "poster_path": data.get("poster_path"),
        "backdrop_path": data.get("backdrop_path"),
        "tagline": data.get("tagline", "")
    }


# Handle individual request failures
failed_ids = []

def safe_movie_details(movie_id):
    try:
        return get_movie_details(movie_id)
    except Exception:
        failed_ids.append(movie_id)
        return None


# Parallel retrieval to reduce processing time
movie_ids = movies_df["id"].tolist()

with ThreadPoolExecutor(max_workers=4) as executor:
    detailed_movies = list(
        tqdm(
            executor.map(safe_movie_details, movie_ids),
            total=len(movie_ids),
            desc="Downloading movie metadata"
        )
    )

df = pd.DataFrame([
    movie for movie in detailed_movies
    if movie is not None
])

print(f"Movies successfully processed: {len(df)}")
print(f"Failed requests: {len(failed_ids)}")
print(f"Total features: {df.shape[1]}")

display(df.head())


In [ ]:

# Section 7: Data Cleaning and Feature Engineering

df = df.drop_duplicates(subset="id").copy()

# Extract release year and decade
df["release_date"] = pd.to_datetime(
    df["release_date"], errors="coerce"
)

df["year"] = df["release_date"].dt.year
df["decade"] = (df["year"] // 10) * 10

# Clean missing text values
for column in ["title", "overview", "tagline"]:
    df[column] = df[column].fillna("")

# Create full poster URLs
df["poster_url"] = df["poster_path"].apply(
    lambda path: IMAGE_BASE_URL + path
    if isinstance(path, str) and path
    else None
)

# Zero budget/revenue/runtime may indicate missing data
for column in ["budget", "revenue", "runtime"]:
    df[column] = pd.to_numeric(
        df[column], errors="coerce"
    ).replace(0, np.nan)

# Weighted rating to reduce low-vote bias
C = df["vote_average"].mean()
m = max(df["vote_count"].quantile(0.60), 1)

v = df["vote_count"]
R = df["vote_average"]

df["weighted_rating"] = (
    (v / (v + m)) * R
    + (m / (v + m)) * C
)

# Combine metadata for content-based recommendations
df["combined_features"] = df.apply(
    lambda row: " ".join([
        row["overview"],
        row["tagline"],
        " ".join(row["genres"] * 2),
        " ".join(row["keywords"]),
        " ".join(row["cast"]),
        " ".join(row["directors"])
    ]),
    axis=1
)

df = df.reset_index(drop=True)

print("Feature engineering completed!")
print(f"Total movies: {len(df)}")
print(f"Total features: {df.shape[1]}")

display(df[[
    "title", "year", "genres", "runtime",
    "certification", "weighted_rating"
]].head(10))


In [ ]:

# Section 8: Save Processed Dataset

df.to_json(
    "cinemood_movies.json",
    orient="records",
    date_format="iso",
    force_ascii=False
)

df.to_csv(
    "cinemood_movies.csv",
    index=False
)

print("Dataset saved successfully!")

# Data quality report
quality_report = pd.DataFrame({
    "Feature": df.columns,
    "Missing Values": df.isna().sum().values,
    "Completeness (%)": (
        df.notna().mean().values * 100
    ).round(2)
})

display(quality_report)


In [ ]:

# Section 9: NLP Sentiment Analysis

sentiment_scores = df["overview"].apply(
    sia.polarity_scores
).apply(pd.Series)

df["sentiment_positive"] = sentiment_scores["pos"]
df["sentiment_negative"] = sentiment_scores["neg"]
df["sentiment_neutral"] = sentiment_scores["neu"]
df["sentiment_compound"] = sentiment_scores["compound"]

# Classify the sentiment expressed in plot descriptions
def classify_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df["sentiment_label"] = df[
    "sentiment_compound"
].apply(classify_sentiment)

print("Sentiment analysis completed!")

display(df[[
    "title",
    "sentiment_positive",
    "sentiment_negative",
    "sentiment_neutral",
    "sentiment_compound",
    "sentiment_label"
]].head(10))

print("\nSentiment Distribution:")
print(df["sentiment_label"].value_counts())


In [ ]:

# Section 10: Multidimensional Mood Intelligence

import re

MOOD_PROFILES = {
    "Feel-good": {
        "genres": ["Comedy", "Family", "Animation"],
        "keywords": [
            "friendship", "happiness", "hope",
            "heartwarming", "joy", "family"
        ],
        "sentiment": 0.7
    },
    "Romantic": {
        "genres": ["Romance"],
        "keywords": [
            "love", "romance", "relationship",
            "marriage", "couple", "fall in love"
        ],
        "sentiment": 0.4
    },
    "Funny": {
        "genres": ["Comedy"],
        "keywords": [
            "humor", "comedy", "funny",
            "hilarious", "satire", "parody"
        ],
        "sentiment": 0.5
    },
    "Exciting": {
        "genres": ["Action", "Adventure"],
        "keywords": [
            "mission", "battle", "chase",
            "escape", "hero", "survival"
        ],
        "sentiment": 0.0
    },
    "Dark & Suspenseful": {
        "genres": ["Horror", "Thriller", "Mystery"],
        "keywords": [
            "murder", "revenge", "fear",
            "dark", "crime", "terror"
        ],
        "sentiment": -0.7
    },
    "Emotional": {
        "genres": ["Drama", "Romance"],
        "keywords": [
            "loss", "grief", "sacrifice",
            "tragedy", "family", "heartbreak"
        ],
        "sentiment": -0.3
    },
    "Inspiring": {
        "genres": ["Drama", "History", "Documentary"],
        "keywords": [
            "dream", "courage", "hope",
            "overcome", "success", "determination"
        ],
        "sentiment": 0.6
    },
    "Thought-provoking": {
        "genres": ["Science Fiction", "Mystery", "Documentary"],
        "keywords": [
            "identity", "philosophy", "society",
            "reality", "consciousness", "morality"
        ],
        "sentiment": 0.0
    }
}


def calculate_mood_score(row, profile):

    # Genre compatibility
    movie_genres = set(row["genres"])
    target_genres = set(profile["genres"])

    genre_score = float(
        bool(movie_genres & target_genres)
    )

    # Keyword compatibility
    text = (
        row["overview"] + " " +
        row["tagline"] + " " +
        " ".join(row["keywords"])
    ).lower()

    keyword_matches = sum(
        bool(re.search(
            r"\b" + re.escape(word) + r"\b",
            text
        ))
        for word in profile["keywords"]
    )

    keyword_score = min(keyword_matches / 3, 1)

    # Sentiment compatibility
    sentiment_score = 1 - (
        abs(
            row["sentiment_compound"]
            - profile["sentiment"]
        ) / 2
    )

    # Combined emotional compatibility
    final_score = (
        0.45 * genre_score
        + 0.35 * keyword_score
        + 0.20 * sentiment_score
    )

    return round(final_score, 4)


for mood, profile in MOOD_PROFILES.items():

    df[f"mood_{mood}"] = df.apply(
        lambda row: calculate_mood_score(
            row, profile
        ),
        axis=1
    )


mood_columns = [
    f"mood_{mood}"
    for mood in MOOD_PROFILES
]

df["dominant_mood"] = (
    df[mood_columns]
    .idxmax(axis=1)
    .str.replace("mood_", "", regex=False)
)

df["dominant_mood_score"] = df[
    mood_columns
].max(axis=1)

print("Mood Intelligence completed!")

display(df[[
    "title",
    "genres",
    "sentiment_label",
    "dominant_mood",
    "dominant_mood_score"
]].head(15))


In [ ]:

# Section 11: Emotional Analytics Dashboard

mood_distribution = (
    df["dominant_mood"]
    .value_counts()
    .rename_axis("Mood")
    .reset_index(name="Movies")
)

fig = px.bar(
    mood_distribution,
    x="Mood",
    y="Movies",
    color="Mood",
    text="Movies",
    title="CineMood - Emotional Movie Distribution",
    template="plotly_dark"
)

fig.update_layout(
    showlegend=False,
    xaxis_title="Emotional Profile",
    yaxis_title="Number of Movies"
)

fig.show()


# Sentiment distribution
sentiment_distribution = (
    df["sentiment_label"]
    .value_counts()
    .rename_axis("Sentiment")
    .reset_index(name="Movies")
)

fig2 = px.pie(
    sentiment_distribution,
    names="Sentiment",
    values="Movies",
    hole=0.55,
    title="Plot Description Sentiment Distribution",
    template="plotly_dark",
    color="Sentiment",
    color_discrete_map={
        "Positive": "#22c55e",
        "Neutral": "#94a3b8",
        "Negative": "#ef4444"
    }
)

fig2.show()


In [ ]:

# Section 12: Content-Based Recommendation Engine

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    max_features=20000,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(
    df["combined_features"]
)

# Calculate pairwise movie similarity
similarity_matrix = cosine_similarity(
    tfidf_matrix
)

print("Content-based model trained!")
print("TF-IDF matrix:", tfidf_matrix.shape)
print("Similarity matrix:", similarity_matrix.shape)


def recommend_similar(movie_title, top_n=10):

    matches = df.index[
        df["title"].str.casefold()
        == movie_title.strip().casefold()
    ].tolist()

    if not matches:
        return pd.DataFrame()

    movie_index = matches[0]

    scores = similarity_matrix[
        movie_index
    ].copy()

    # Exclude the selected movie
    scores[movie_index] = -1

    ranked_indices = np.argsort(
        scores
    )[::-1][:top_n]

    results = df.iloc[
        ranked_indices
    ][[
        "title",
        "year",
        "genres",
        "vote_average",
        "dominant_mood"
    ]].copy()

    results["content_similarity"] = (
        scores[ranked_indices] * 100
    ).round(2)

    return results.reset_index(drop=True)


# Test the recommendation engine
display(recommend_similar("Iron Man", 10))


In [ ]:

# Section 13: Sentiment-Based Recommendation Engine

def recommend_by_sentiment(
    target_sentiment,
    top_n=10
):

    target_sentiment = float(
        np.clip(target_sentiment, -1, 1)
    )

    results = df.copy()

    results["sentiment_similarity"] = (
        1 - abs(
            results["sentiment_compound"]
            - target_sentiment
        ) / 2
    )

    results = results.sort_values(
        ["sentiment_similarity", "weighted_rating"],
        ascending=False
    )

    return results[[
        "title",
        "genres",
        "sentiment_compound",
        "sentiment_similarity",
        "weighted_rating"
    ]].head(top_n)


# Test positive sentiment recommendations
display(recommend_by_sentiment(0.8, 10))


In [ ]:

# Section 14: Save Enriched Dataset

df.to_json(
    "cinemood_movies.json",
    orient="records",
    date_format="iso",
    force_ascii=False
)

df.to_csv(
    "cinemood_movies.csv",
    index=False
)

print("Enriched dataset saved successfully!")
print(f"Total movies: {len(df)}")
print(f"Total features: {df.shape[1]}")


In [ ]:

# Section 15: Advanced Filtering System

def filter_movies(
    year_range=None,
    release_period=None,
    min_rating=0,
    min_votes=0,
    runtime_range=None,
    certifications=None,
    languages=None,
    actors=None,
    directors=None,
    genres=None,
    genre_operator="AND",
    excluded_genres=None
):

    results = df.copy()

    # Temporal filters
    if year_range is not None:
        results = results[
            results["year"].between(*year_range)
        ]

    if release_period is not None:
        start, end = release_period

        results = results[
            results["release_date"].between(
                pd.Timestamp(start),
                pd.Timestamp(end)
            )
        ]

    # Quality filters
    results = results[
        (results["vote_average"] >= min_rating)
        & (results["vote_count"] >= min_votes)
    ]

    # Runtime filter
    if runtime_range is not None:
        results = results[
            results["runtime"].between(
                *runtime_range
            )
        ]

    # Certification and language
    if certifications:
        results = results[
            results["certification"].isin(
                certifications
            )
        ]

    if languages:
        results = results[
            results["original_language"].isin(
                languages
            )
        ]

    # Actor preferences
    if actors:
        selected = {
            name.casefold() for name in actors
        }

        results = results[
            results["cast"].apply(
                lambda names: bool(
                    selected & {
                        n.casefold() for n in names
                    }
                )
            )
        ]

    # Director preferences
    if directors:
        selected = {
            name.casefold() for name in directors
        }

        results = results[
            results["directors"].apply(
                lambda names: bool(
                    selected & {
                        n.casefold() for n in names
                    }
                )
            )
        ]

    # Genre logic: AND / OR
    if genres:

        selected_genres = set(genres)

        if genre_operator == "AND":
            results = results[
                results["genres"].apply(
                    lambda x:
                    selected_genres.issubset(set(x))
                )
            ]

        elif genre_operator == "OR":
            results = results[
                results["genres"].apply(
                    lambda x: bool(
                        selected_genres & set(x)
                    )
                )
            ]

        else:
            raise ValueError(
                "Genre operator must be AND or OR."
            )

    # Genre exclusion: NOT
    if excluded_genres:
        excluded = set(excluded_genres)

        results = results[
            results["genres"].apply(
                lambda x: not bool(
                    excluded & set(x)
                )
            )
        ]

    return results.copy()


# Test combined filtering
filtered = filter_movies(
    year_range=(2000, 2026),
    min_rating=7,
    min_votes=500,
    runtime_range=(60, 130),
    genres=["Comedy", "Romance"],
    genre_operator="AND",
    excluded_genres=["Horror"]
)

print(f"Matching movies: {len(filtered)}")

display(filtered[[
    "title", "year", "genres",
    "runtime", "vote_average"
]].head(10))


In [ ]:

# Section 16: Natural Language Query Processing

GENRE_ALIASES = {
    "Action": ["action"],
    "Adventure": ["adventure"],
    "Comedy": ["comedy", "funny", "hilarious"],
    "Romance": ["romance", "romantic"],
    "Horror": ["horror", "scary"],
    "Thriller": ["thriller", "suspense"],
    "Drama": ["drama"],
    "Science Fiction": ["science fiction", "sci-fi"],
    "Fantasy": ["fantasy"],
    "Animation": ["animation", "animated"],
    "Family": ["family"],
    "Crime": ["crime"],
    "Mystery": ["mystery"],
    "Documentary": ["documentary"],
    "History": ["history", "historical"],
    "Music": ["music", "musical"],
    "War": ["war"],
    "Western": ["western"]
}

MOOD_ALIASES = {
    "Feel-good": [
        "feel-good", "feel good", "happy",
        "cheerful", "heartwarming", "comforting"
    ],
    "Romantic": ["romantic", "romance", "love"],
    "Funny": ["funny", "hilarious", "laugh"],
    "Exciting": ["exciting", "adrenaline", "action-packed"],
    "Dark & Suspenseful": [
        "dark", "suspenseful", "scary"
    ],
    "Emotional": ["emotional", "sad", "moving"],
    "Inspiring": ["inspiring", "motivational", "uplifting"],
    "Thought-provoking": [
        "thought-provoking", "intellectual",
        "philosophical", "mind-bending"
    ]
}


def contains_phrase(text, phrase):
    return bool(re.search(
        r"\b" + re.escape(phrase) + r"\b",
        text
    ))


def parse_query(query):

    text = query.lower().strip()

    preferences = {
        "genres": [],
        "excluded_genres": [],
        "genre_operator": "AND",
        "mood": None,
        "runtime_range": None,
        "min_rating": 0,
        "year_range": None
    }

    positive_text = text

    # Detect explicit genre exclusions
    for genre, aliases in GENRE_ALIASES.items():

        for alias in aliases:

            pattern = (
                r"\b(?:no|not|without|exclude|avoid)"
                r"\s+(?:any\s+)?"
                + re.escape(alias)
                + r"\b"
            )

            if re.search(pattern, text):
                preferences["excluded_genres"].append(
                    genre
                )

                positive_text = re.sub(
                    pattern, " ", positive_text
                )

    # Detect requested genres
    for genre, aliases in GENRE_ALIASES.items():

        if genre not in preferences["excluded_genres"]:

            if any(
                contains_phrase(positive_text, alias)
                for alias in aliases
            ):
                preferences["genres"].append(genre)

    # Detect genre alternatives
    if contains_phrase(positive_text, "or"):
        preferences["genre_operator"] = "OR"

    # Identify strongest matching mood
    mood_matches = {
        mood: sum(
            contains_phrase(positive_text, alias)
            for alias in aliases
        )
        for mood, aliases in MOOD_ALIASES.items()
    }

    if max(mood_matches.values()) > 0:
        preferences["mood"] = max(
            mood_matches, key=mood_matches.get
        )

    # Detect maximum runtime
    runtime_match = re.search(
        r"(under|less than|at most|up to|maximum)"
        r"\s+(\d+(?:\.\d+)?)"
        r"\s*(minutes?|mins?|hours?|hrs?)\b",
        text
    )

    if runtime_match:

        operator = runtime_match.group(1)
        duration = float(runtime_match.group(2))
        unit = runtime_match.group(3)

        if unit.startswith(("hour", "hr")):
            duration *= 60

        if operator in ["under", "less than"]:
            duration = int(np.ceil(duration)) - 1
        else:
            duration = int(duration)

        preferences["runtime_range"] = (
            1, duration
        )

    # Detect minimum rating
    rating_match = re.search(
        r"(?:rated|rating)\s*"
        r"(?:above|over|at least|of|>=)?"
        r"\s*(\d+(?:\.\d+)?)",
        text
    )

    if rating_match:
        rating = float(rating_match.group(1))

        if 0 <= rating <= 10:
            preferences["min_rating"] = rating

    # Detect decade
    decade_match = re.search(
        r"\b((?:19|20)\d0)s\b",
        text
    )

    if decade_match:
        decade = int(decade_match.group(1))

        preferences["year_range"] = (
            decade, decade + 9
        )

    # Detect explicit year range
    year_match = re.search(
        r"\b((?:19|20)\d{2})\s*"
        r"(?:to|-)\s*((?:19|20)\d{2})\b",
        text
    )

    if year_match:
        preferences["year_range"] = tuple(
            sorted(map(int, year_match.groups()))
        )

    preferences["excluded_genres"] = list(
        dict.fromkeys(
            preferences["excluded_genres"]
        )
    )

    return preferences


# Test natural language processing
query = (
    "I want a funny romantic movie "
    "under 120 minutes, rated above 7, "
    "without horror."
)

parsed_preferences = parse_query(query)

print("Original query:")
print(query)

print("\nExtracted preferences:")

for key, value in parsed_preferences.items():
    print(f"{key}: {value}")


In [ ]:

# Section 17: Adaptive Hybrid Recommendation Engine

def hybrid_recommend(
    query="",
    mood=None,
    reference_title=None,
    top_n=10,
    filters=None,
    weights=(0.50, 0.35, 0.15)
):

    filters = filters or {}

    candidates = filter_movies(**filters)

    if candidates.empty:
        return candidates

    # Initialize component scores
    content_scores = np.zeros(len(df))
    mood_scores = np.zeros(len(df))

    active_weights = np.array(
        weights, dtype=float
    )

    if (
        active_weights.shape != (3,)
        or not np.isfinite(active_weights).all()
        or (active_weights < 0).any()
    ):
        raise ValueError("Invalid model weights.")

    content_available = False

    # Content similarity from a reference movie
    if reference_title:

        matches = df.index[
            df["title"].str.casefold()
            == reference_title.strip().casefold()
        ].tolist()

        if not matches:
            raise ValueError(
                "Reference movie not found."
            )

        reference_index = matches[0]

        content_scores = similarity_matrix[
            reference_index
        ].copy()

        candidates = candidates.drop(
            index=reference_index,
            errors="ignore"
        )

        content_available = True

    # Content similarity from natural language
    if query.strip():

        query_vector = tfidf.transform([query])

        if query_vector.nnz > 0:

            query_scores = cosine_similarity(
                query_vector, tfidf_matrix
            ).flatten()

            if content_available:
                content_scores = (
                    0.60 * content_scores
                    + 0.40 * query_scores
                )
            else:
                content_scores = query_scores

            content_available = True

    if not content_available:
        active_weights[0] = 0

    # Mood compatibility
    if mood in MOOD_PROFILES:

        mood_scores = df[
            f"mood_{mood}"
        ].to_numpy()

    else:
        active_weights[1] = 0

    # Normalize active weights
    if active_weights.sum() == 0:
        active_weights[2] = 1

    active_weights /= active_weights.sum()

    # Quality score
    quality_scores = (
        df["weighted_rating"]
        .fillna(0)
        .to_numpy() / 10
    )

    # Apply scores to filtered candidates
    indices = candidates.index.to_numpy()

    candidates["content_score"] = (
        content_scores[indices]
    )

    candidates["mood_score"] = (
        mood_scores[indices]
    )

    candidates["quality_score"] = (
        quality_scores[indices]
    )

    candidates["hybrid_score"] = (
        active_weights[0]
        * candidates["content_score"]
        + active_weights[1]
        * candidates["mood_score"]
        + active_weights[2]
        * candidates["quality_score"]
    )

    candidates["match_score"] = (
        candidates["hybrid_score"] * 100
    ).round(2)

    candidates["content_weight"] = active_weights[0]
    candidates["mood_weight"] = active_weights[1]
    candidates["quality_weight"] = active_weights[2]

    return candidates.sort_values(
        ["hybrid_score", "weighted_rating"],
        ascending=False
    ).head(top_n).copy()


# Test the hybrid model
hybrid_results = hybrid_recommend(
    query="superhero technology adventure",
    mood="Exciting",
    reference_title="Iron Man",
    filters={
        "min_rating": 6.5,
        "min_votes": 500
    },
    top_n=10
)

display(hybrid_results[[
    "title",
    "content_score",
    "mood_score",
    "quality_score",
    "match_score"
]])


In [ ]:

# Section 18: Explainable AI

def explain_recommendation(
    row,
    mood=None,
    reference_title=None,
    requested_genres=None
):

    reasons = []

    # Explain genre compatibility
    if requested_genres:

        shared_genres = sorted(
            set(row["genres"])
            & set(requested_genres)
        )

        if shared_genres:
            reasons.append(
                "Matches your preferred genres: "
                + ", ".join(shared_genres)
            )

    # Explain content similarity
    if row["content_weight"] > 0:

        score = row["content_score"] * 100

        if reference_title:
            reasons.append(
                f"Content similarity to "
                f"{reference_title}: {score:.1f}%"
            )
        else:
            reasons.append(
                f"Text similarity to your query: "
                f"{score:.1f}%"
            )

    # Explain emotional compatibility
    if mood and row["mood_weight"] > 0:

        score = row["mood_score"] * 100

        reasons.append(
            f"{mood} mood compatibility: "
            f"{score:.1f}/100"
        )

    # Explain movie quality
    reasons.append(
        f"TMDB rating: "
        f"{row['vote_average']:.1f}/10 "
        f"from {row['vote_count']:,} votes"
    )

    # Explain duration
    if pd.notna(row["runtime"]):

        reasons.append(
            f"Runtime: {int(row['runtime'])} minutes"
        )

    # Explain the largest ranking contribution
    contributions = {
        "Content similarity":
            row["content_score"]
            * row["content_weight"],

        "Mood compatibility":
            row["mood_score"]
            * row["mood_weight"],

        "Weighted rating":
            row["quality_score"]
            * row["quality_weight"]
    }

    main_factor = max(
        contributions,
        key=contributions.get
    )

    reasons.append(
        f"Main ranking factor: {main_factor}"
    )

    return " | ".join(reasons)


hybrid_results["explanation"] = (
    hybrid_results.apply(
        lambda row: explain_recommendation(
            row,
            mood="Exciting",
            reference_title="Iron Man"
        ),
        axis=1
    )
)

for _, row in hybrid_results.head(5).iterrows():

    print(f"\nMOVIE: {row['title']}")
    print(f"Match Score: {row['match_score']}/100")
    print(row["explanation"])
    print("-" * 80)


In [ ]:

# Section 19: Integrated Intelligent Search

def intelligent_search(query, top_n=10):

    preferences = parse_query(query)

    mood = preferences.pop("mood")

    results = hybrid_recommend(
        query=query,
        mood=mood,
        filters=preferences,
        top_n=top_n
    )

    if results.empty:

        print(
            "No movies match all your preferences. "
            "Try adjusting your filters."
        )

        return results

    results["explanation"] = results.apply(
        lambda row: explain_recommendation(
            row,
            mood=mood,
            requested_genres=preferences["genres"]
        ),
        axis=1
    )

    print("Detected preferences:")
    print(preferences)
    print(f"Detected mood: {mood}")

    return results


# End-to-end system test
results = intelligent_search(
    "I want a funny romantic movie "
    "under 120 minutes, rated above 7, "
    "without horror.",
    top_n=10
)

if not results.empty:

    display(results[[
        "title",
        "year",
        "genres",
        "runtime",
        "vote_average",
        "match_score",
        "explanation"
    ]])


In [ ]:

# Section 20: NLP Boundary Improvements

original_parse_query = parse_query

def parse_query(query):

    preferences = original_parse_query(query)

    text = query.lower().strip()

    # Distinguish strict and inclusive rating thresholds
    strict_rating = re.search(
        r"(?:rated|rating)\s+(?:above|over)"
        r"\s+(\d+(?:\.\d+)?)",
        text
    )

    if strict_rating:
        threshold = float(strict_rating.group(1))

        if 0 <= threshold <= 10:
            preferences["min_rating"] = np.nextafter(
                threshold, np.inf
            )

    return preferences


print("NLP boundary improvements completed!")

print(parse_query(
    "Comedy rated above 7 under 2 hours"
))


In [ ]:

# Section 21: Automated Quality Assurance

test_results = []

def run_test(name, test_function):

    try:
        passed = bool(test_function())
        status = "PASS" if passed else "FAIL"
        details = ""

    except Exception as error:
        status = "FAIL"
        details = str(error)

    test_results.append({
        "Test": name,
        "Status": status,
        "Details": details
    })


def validate_filter(filters, condition):

    output = filter_movies(**filters)

    return (
        not output.empty
        and bool(condition(output).all())
    )


# 1. Data integrity
run_test(
    "Unique movie IDs",
    lambda: df["id"].is_unique
)

run_test(
    "Valid rating range",
    lambda: df["vote_average"].between(0, 10).all()
)

run_test(
    "Valid sentiment range",
    lambda: df["sentiment_compound"].between(-1, 1).all()
)

run_test(
    "Valid mood scores",
    lambda: df[mood_columns].apply(
        lambda x: x.between(0, 1).all()
    ).all()
)

# 2. Temporal filters
run_test(
    "Year range filter",
    lambda: validate_filter(
        {"year_range": (1990, 1999)},
        lambda x: x["year"].between(1990, 1999)
    )
)

run_test(
    "Release period filter",
    lambda: validate_filter(
        {
            "release_period":
                ("2010-01-01", "2019-12-31")
        },
        lambda x: x["release_date"].between(
            "2010-01-01", "2019-12-31"
        )
    )
)

# 3. Quality filters
run_test(
    "Minimum rating filter",
    lambda: validate_filter(
        {"min_rating": 8},
        lambda x: x["vote_average"] >= 8
    )
)

run_test(
    "Minimum vote count filter",
    lambda: validate_filter(
        {"min_votes": 1000},
        lambda x: x["vote_count"] >= 1000
    )
)

# 4. Content specifications
run_test(
    "Runtime filter",
    lambda: validate_filter(
        {"runtime_range": (60, 120)},
        lambda x: x["runtime"].between(60, 120)
    )
)

run_test(
    "Certification filter",
    lambda: validate_filter(
        {"certifications": ["PG", "PG-13"]},
        lambda x: x["certification"].isin(
            ["PG", "PG-13"]
        )
    )
)

run_test(
    "Language filter",
    lambda: validate_filter(
        {"languages": ["en"]},
        lambda x: x["original_language"] == "en"
    )
)

# 5. Personnel filters
run_test(
    "Actor filter",
    lambda: validate_filter(
        {"actors": ["Robert Downey Jr."]},
        lambda x: x["cast"].apply(
            lambda a: "Robert Downey Jr." in a
        )
    )
)

run_test(
    "Director filter",
    lambda: validate_filter(
        {"directors": ["Christopher Nolan"]},
        lambda x: x["directors"].apply(
            lambda a: "Christopher Nolan" in a
        )
    )
)

# 6. Genre logic
run_test(
    "Genre AND logic",
    lambda: validate_filter(
        {
            "genres": ["Comedy", "Romance"],
            "genre_operator": "AND"
        },
        lambda x: x["genres"].apply(
            lambda g: {"Comedy", "Romance"}.issubset(g)
        )
    )
)

run_test(
    "Genre OR logic",
    lambda: validate_filter(
        {
            "genres": ["Comedy", "Action"],
            "genre_operator": "OR"
        },
        lambda x: x["genres"].apply(
            lambda g: bool({"Comedy", "Action"} & set(g))
        )
    )
)

run_test(
    "Genre NOT logic",
    lambda: validate_filter(
        {"excluded_genres": ["Horror"]},
        lambda x: x["genres"].apply(
            lambda g: "Horror" not in g
        )
    )
)

# 7. Combined constraints
run_test(
    "Combined filter compliance",
    lambda: validate_filter(
        {
            "year_range": (2000, 2026),
            "min_rating": 7,
            "min_votes": 500,
            "runtime_range": (60, 130),
            "genres": ["Comedy", "Romance"],
            "excluded_genres": ["Horror"]
        },
        lambda x: (
            x["year"].between(2000, 2026)
            & (x["vote_average"] >= 7)
            & (x["vote_count"] >= 500)
            & x["runtime"].between(60, 130)
            & x["genres"].apply(
                lambda g:
                    {"Comedy", "Romance"}.issubset(g)
                    and "Horror" not in g
            )
        )
    )
)

# 8. Natural language processing
run_test(
    "NLP runtime conversion",
    lambda: parse_query(
        "Comedy under 2 hours"
    )["runtime_range"] == (1, 119)
)

run_test(
    "NLP genre exclusion",
    lambda: "Horror" in parse_query(
        "Comedy without horror"
    )["excluded_genres"]
)

run_test(
    "NLP decade recognition",
    lambda: parse_query(
        "Action movies from the 1990s"
    )["year_range"] == (1990, 1999)
)

run_test(
    "NLP strict rating threshold",
    lambda: parse_query(
        "Rated above 7"
    )["min_rating"] > 7
)

# 9. Edge cases
run_test(
    "Impossible search returns empty results",
    lambda: filter_movies(
        min_rating=11
    ).empty
)

run_test(
    "Unknown reference title",
    lambda: recommend_similar(
        "NonexistentMovieXYZ"
    ).empty
)

run_test(
    "Reference movie excluded from results",
    lambda: "Iron Man" not in
        recommend_similar("Iron Man")["title"].values
)

run_test(
    "Hybrid score within valid range",
    lambda: hybrid_results[
        "match_score"
    ].between(0, 100).all()
)

run_test(
    "Hybrid ranking descending",
    lambda: hybrid_results[
        "hybrid_score"
    ].is_monotonic_decreasing
)

# Generate quality report
qa_report = pd.DataFrame(test_results)

passed = (qa_report["Status"] == "PASS").sum()
total = len(qa_report)

print(f"Tests passed: {passed}/{total}")
print(f"Test pass rate: {passed / total * 100:.2f}%")

display(qa_report)

qa_report.to_csv(
    "cinemood_quality_report.csv",
    index=False
)


In [ ]:

# Section 22: Comparative Algorithm Evaluation

from itertools import combinations
from time import perf_counter

def genre_similarity(genres_a, genres_b):

    a = set(genres_a)
    b = set(genres_b)

    union = a | b

    return len(a & b) / len(union) if union else 0


def evaluate_algorithms(reference_title, top_n=10):

    reference_index = df.index[
        df["title"].str.casefold()
        == reference_title.casefold()
    ][0]

    reference = df.loc[reference_index]

    mood = reference["dominant_mood"]

    candidates = df.drop(
        index=reference_index
    ).copy()

    indices = candidates.index.to_numpy()

    model_scores = {
        "Popularity Baseline":
            candidates["popularity"].to_numpy(),

        "Content-Based":
            similarity_matrix[
                reference_index, indices
            ],

        "Sentiment-Based":
            1 - abs(
                candidates["sentiment_compound"].to_numpy()
                - reference["sentiment_compound"]
            ) / 2,

        "Hybrid":
            0.50 * similarity_matrix[
                reference_index, indices
            ]
            + 0.35 * candidates[
                f"mood_{mood}"
            ].to_numpy()
            + 0.15 * candidates[
                "weighted_rating"
            ].to_numpy() / 10
    }

    evaluation = []

    for model, scores in model_scores.items():

        start = perf_counter()

        ranked = candidates.assign(
            model_score=scores
        ).nlargest(top_n, "model_score")

        ranking_time = (
            perf_counter() - start
        ) * 1000

        # Genre relevance proxy
        relevance = ranked["genres"].apply(
            lambda g: genre_similarity(
                g, reference["genres"]
            )
        ).mean()

        # Intra-list diversity
        pairs = list(
            combinations(ranked.index, 2)
        )

        diversity = np.mean([
            1 - similarity_matrix[i, j]
            for i, j in pairs
        ]) if pairs else 0

        evaluation.append({
            "Reference": reference_title,
            "Model": model,
            "Genre Relevance": relevance,
            "Mood Compatibility":
                ranked[f"mood_{mood}"].mean(),
            "Average Rating":
                ranked["vote_average"].mean(),
            "Diversity": diversity,
            "Ranking Time (ms)": ranking_time
        })

    return pd.DataFrame(evaluation)


# Select reference movies from different genres
reference_movies = [
    "Iron Man",
    "La La Land",
    "Shrek 2",
    "Whiplash",
    "The Proposal",
    "The Lord of the Rings: The Return of the King"
]

available_references = [
    title for title in reference_movies
    if title in df["title"].values
]

evaluation_df = pd.concat(
    [
        evaluate_algorithms(title)
        for title in available_references
    ],
    ignore_index=True
)

evaluation_summary = (
    evaluation_df.groupby("Model")
    .mean(numeric_only=True)
    .round(4)
    .reset_index()
)

print(
    f"Reference movies evaluated: "
    f"{len(available_references)}"
)

display(evaluation_summary)

evaluation_df.to_csv(
    "cinemood_model_evaluation.csv",
    index=False
)


In [ ]:

# Section 23: Model Performance Visualization

comparison = evaluation_summary.melt(
    id_vars="Model",
    value_vars=[
        "Genre Relevance",
        "Mood Compatibility",
        "Diversity"
    ],
    var_name="Metric",
    value_name="Score"
)

fig = px.bar(
    comparison,
    x="Metric",
    y="Score",
    color="Model",
    barmode="group",
    title="CineMood - Recommendation Model Benchmark",
    template="plotly_dark",
    text_auto=".2f",
    color_discrete_sequence=[
        "#6366F1",
        "#A855F7",
        "#14B8A6",
        "#F59E0B"
    ]
)

fig.update_layout(
    yaxis_range=[0, 1.1],
    xaxis_title="Evaluation Metric",
    yaxis_title="Normalized Score",
    legend_title="Recommendation Model"
)

fig.show()


In [ ]:

# Section 24: Enhanced Data Quality Assessment

def is_missing(value):

    if isinstance(value, (list, tuple, dict)):
        return len(value) == 0

    if isinstance(value, str):
        return not value.strip()

    return bool(pd.isna(value))


quality_records = []

for column in df.columns:

    missing = df[column].apply(is_missing).sum()

    quality_records.append({
        "Feature": column,
        "Missing or Empty": missing,
        "Completeness (%)": round(
            (1 - missing / len(df)) * 100,
            2
        )
    })

enhanced_quality_report = pd.DataFrame(
    quality_records
)

display(enhanced_quality_report)

print(
    "Overall data completeness:",
    round(
        enhanced_quality_report[
            "Completeness (%)"
        ].mean(),
        2
    ),
    "%"
)

enhanced_quality_report.to_csv(
    "cinemood_data_quality.csv",
    index=False
)


In [ ]:

# Section 25: Export Production Model Assets

import os
import joblib
import shutil
import sklearn
import scipy
import json
from datetime import datetime, timezone

os.makedirs("cinemood_assets", exist_ok=True)

# Validate dataset and model alignment
assert df.index.tolist() == list(range(len(df)))
assert tfidf_matrix.shape[0] == len(df)
assert similarity_matrix.shape == (len(df), len(df))
assert df["id"].is_unique

# Export dataset
df.to_json(
    "cinemood_assets/cinemood_movies.json",
    orient="records",
    date_format="iso",
    force_ascii=False
)

# Export trained vectorizer
joblib.dump(
    tfidf,
    "cinemood_assets/tfidf_vectorizer.joblib",
    compress=3
)

# Export TF-IDF matrix
joblib.dump(
    tfidf_matrix,
    "cinemood_assets/tfidf_matrix.joblib",
    compress=3
)

# Export similarity matrix
np.save(
    "cinemood_assets/similarity_matrix.npy",
    similarity_matrix.astype(np.float32)
)

# Export evaluation reports
qa_report.to_csv(
    "cinemood_assets/quality_report.csv",
    index=False
)

evaluation_summary.to_csv(
    "cinemood_assets/model_evaluation.csv",
    index=False
)

enhanced_quality_report.to_csv(
    "cinemood_assets/data_quality.csv",
    index=False
)

# Export project metadata
metadata = {
    "project": "CineMood",
    "version": "1.0.0",
    "total_movies": len(df),
    "total_features": df.shape[1],
    "tfidf_features": tfidf_matrix.shape[1],
    "tests_passed": int(
        (qa_report["Status"] == "PASS").sum()
    ),
    "total_tests": len(qa_report),
    "data_source": "The Movie Database (TMDB)",
    "exported_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "library_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "scipy": scipy.__version__,
        "joblib": joblib.__version__
    }
}

with open(
    "cinemood_assets/metadata.json",
    "w"
) as file:
    json.dump(metadata, file, indent=4)

# Display exported files
print("CINEMOOD PRODUCTION ASSETS")
print("=" * 50)

for filename in os.listdir("cinemood_assets"):

    path = os.path.join(
        "cinemood_assets", filename
    )

    size = os.path.getsize(path) / 1024

    print(f"{filename}: {size:.2f} KB")

print("\nAll assets exported successfully!")


In [ ]:

# Section 26: Package Production Assets

from google.colab import files

shutil.make_archive(
    "CineMood_Assets",
    "zip",
    "cinemood_assets"
)

print("Production package ready!")

files.download("CineMood_Assets.zip")
